# Profiling Basics

This notebook shows a classic mistake: loading the model on every request.

Goals:
- use `cProfile` to inspect a slow function
- compare a slow and fast implementation
- see why measuring beats guessing


In [ ]:
import cProfile
import io
import joblib
import timeit
from pathlib import Path

import pandas as pd
from sklearn.datasets import load_digits
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

artifacts = Path("/tmp/week11_profiling")
artifacts.mkdir(exist_ok=True)

digits = load_digits()
X_train, X_test, y_train, y_test = train_test_split(
    digits.data, digits.target, test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

joblib.dump(model, artifacts / "model.pkl")
pd.DataFrame(X_train).to_csv(artifacts / "data.csv", index=False)

sample = X_test[0]


In [ ]:
def slow_predict(sample):
    data = pd.read_csv(artifacts / "data.csv")
    model = joblib.load(artifacts / "model.pkl")
    return model.predict([sample])[0], data.shape


cached_model = joblib.load(artifacts / "model.pkl")


def fast_predict(sample):
    return cached_model.predict([sample])[0]


profile_buffer = io.StringIO()
profiler = cProfile.Profile()
profiler.enable()
slow_predict(sample)
profiler.disable()
profiler.print_stats(sort="cumulative")


In [ ]:
slow_time = timeit.timeit(lambda: slow_predict(sample), number=10) / 10
fast_time = timeit.timeit(lambda: fast_predict(sample), number=1000) / 1000

print(f"Slow version: {slow_time * 1000:.2f} ms")
print(f"Fast version: {fast_time * 1000:.2f} ms")
print(f"Speedup: {slow_time / fast_time:.1f}x")
